# Step 4.5. Clustering Visualization / Diagnostics

이 노트북은 Step 4에서 선택한 최종 clustering 결과를 **시각적으로 점검**하기 위한 단계입니다.

- 입력: `data/analysis/clustering_results/`에 저장된 Step 4 결과
- 출력: `data/analysis/clustering_visualizations/`에 저장되는 PCA, 3D direct view, LDA 시각화
- 주의: 이 단계는 segment 해석을 완성하는 단계가 아닙니다. Segment별 이탈률, 전략, profiling heatmap은 Step 5에서 수행합니다.

UMAP은 제외했습니다. UMAP은 고객 행동 공간이 연속적으로 분포할 때 같은 cluster가 여러 조각으로 흩어져 보일 수 있어, 이번 제출용 흐름에서는 PCA, direct feature view, LDA projection 중심으로 구성합니다.

## 0. 분석 단계 구분

현재 프로젝트는 다음 구조로 관리합니다.

| 단계 | 목적 |
|---|---|
| Step 3.5 | Step4/Step5용 데이터셋 생성 |
| Step 4 | Clustering 후보 비교 및 최종 모델 선택 |
| Step 4.5 | 선택된 cluster 구조를 시각적으로 점검 |
| Step 5 | Segment profiling, 이탈률 분석, 전략 제안 |

이 노트북은 **Step 4.5**에 해당합니다. 따라서 cluster별 비즈니스 해석은 최소화하고, cluster가 feature space에서 어떻게 배치되는지 확인하는 데 집중합니다.

In [ ]:
# ============================================================
# 1. Import & Project Root 설정
# ============================================================

from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    """
    현재 위치가 project root이든 notebooks 폴더이든,
    src/common.py를 기준으로 프로젝트 루트를 찾는다.
    """
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)
    for path in candidates:
        if (path / "src" / "common.py").exists():
            return path
    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src.common import (
    read_csv_safely,
    configure_korean_font,
    get_plotly_layout,
    save_json,
    save_csv,
)

configure_korean_font(verbose=True)

RANDOM_STATE = 42

print("Project root:", ROOT)

## 1. Step 4 결과 파일 로드

이 노트북은 Step 4에서 생성된 결과 파일을 사용합니다. 필수 파일은 다음입니다.

- `09_selected_model.json`
- `10_selected_cluster_labels.csv`
- `11_df_cluster_input_used.csv`
- `12_df_profile_table_used.csv`

위 파일들이 없으면 Step 4 노트북을 먼저 실행해야 합니다.

In [ ]:
# ============================================================
# 2. 경로 설정 및 데이터 로드
# ============================================================

input_dir = ROOT / "data" / "analysis" / "clustering_results"
output_dir = ROOT / "data" / "analysis" / "clustering_visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

selected_model_path = input_dir / "09_selected_model.json"
selected_labels_path = input_dir / "10_selected_cluster_labels.csv"
df_cluster_path = input_dir / "11_df_cluster_input_used.csv"
df_profile_path = input_dir / "12_df_profile_table_used.csv"

required_paths = [
    selected_model_path,
    selected_labels_path,
    df_cluster_path,
    df_profile_path,
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Step 4 결과 파일이 없습니다. 먼저 04_clustering_model_selection.ipynb를 실행하세요.\n"
        + "\n".join(str(path.relative_to(ROOT)) for path in missing_paths)
    )

with open(selected_model_path, "r", encoding="utf-8") as f:
    selected_model_info = json.load(f)

selected_key = selected_model_info["selected_key"]
selected_features = selected_model_info.get("selected_features", [])

df_cluster = read_csv_safely(df_cluster_path)
df_profile = read_csv_safely(df_profile_path)
selected_labels = read_csv_safely(selected_labels_path)

print("Selected model:", selected_key)
print("Selected features:", selected_features)
print("df_cluster:", df_cluster.shape)
print("df_profile:", df_profile.shape)
print("selected_labels:", selected_labels.shape)

display(pd.DataFrame([selected_model_info.get("selected_metrics", {})]))

## 2. 시각화용 테이블 생성

`df_cluster`는 clustering input으로 사용된 행동 축 데이터를 담고 있고, `selected_labels`는 최종 cluster label을 담고 있습니다. 두 파일은 `Row_ID`를 기준으로 결합합니다.

Step 4.5에서는 segment 이름을 확정하지 않고 `Cluster 0`, `Cluster 1`처럼 중립적인 이름을 사용합니다. 최종 segment 이름은 Step 5에서 profile을 해석한 뒤 확정합니다.

In [ ]:
# ============================================================
# 3. 시각화용 테이블 생성
# ============================================================

if "Row_ID" not in df_cluster.columns:
    raise ValueError("df_cluster에 Row_ID가 없습니다.")

if "Row_ID" not in selected_labels.columns or "Cluster" not in selected_labels.columns:
    raise ValueError("selected_labels에는 Row_ID와 Cluster 컬럼이 있어야 합니다.")

# selected feature가 비어 있거나 누락된 경우 fallback 처리
if not selected_features:
    selected_features = [col for col in df_cluster.columns if col != "Row_ID"]

missing_selected_features = [col for col in selected_features if col not in df_cluster.columns]
if missing_selected_features:
    raise ValueError(f"선택된 feature가 df_cluster에 없습니다: {missing_selected_features}")

# Row_ID 기준 결합
vis_base = df_cluster.merge(selected_labels, on="Row_ID", how="left")

if vis_base["Cluster"].isnull().any():
    raise ValueError("일부 Row_ID에 cluster label이 없습니다.")

vis_base["Cluster"] = vis_base["Cluster"].astype(int)
vis_base["Cluster_Label"] = vis_base["Cluster"].apply(lambda x: f"Cluster {x}")

# hover 정보용 profile 변수 일부만 결합
profile_hover_cols = [
    "Row_ID",
    "Attrition_Label",
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Avg_Utilization_Ratio",
    "High_Risk_Signal_Count",
]

profile_hover_cols = [col for col in profile_hover_cols if col in df_profile.columns]

if "Row_ID" in df_profile.columns:
    df_vis = vis_base.merge(df_profile[profile_hover_cols], on="Row_ID", how="left")
else:
    # Row_ID가 없다면 row order를 기준으로 붙인다.
    df_vis = vis_base.copy()
    for col in profile_hover_cols:
        if col != "Row_ID":
            df_vis[col] = df_profile[col].values

save_csv(df_vis, output_dir / "selected_model_visualization_table.csv")

print("df_vis:", df_vis.shape)
print("Clusters:", sorted(df_vis["Cluster"].unique()))
display(df_vis.head())

## 3. Cluster size 점검

이 단계는 시각화 전에 cluster가 지나치게 불균형하지 않은지 확인하는 간단한 진단입니다. 너무 작은 cluster가 있으면 발표용 segment로 쓰기 어려울 수 있습니다.

In [ ]:
# ============================================================
# 4. Cluster size summary
# ============================================================

cluster_size = (
    df_vis.groupby(["Cluster", "Cluster_Label"])
    .size()
    .reset_index(name="n_customers")
)
cluster_size["cluster_ratio"] = cluster_size["n_customers"] / cluster_size["n_customers"].sum()
cluster_size["cluster_ratio_percent"] = cluster_size["cluster_ratio"] * 100

save_csv(cluster_size, output_dir / "01_cluster_size_summary.csv")

display(cluster_size)

# Matplotlib static chart
plt.figure(figsize=(9, 5))
bars = plt.bar(cluster_size["Cluster_Label"], cluster_size["n_customers"])
plt.title("Cluster Size Summary")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)

for bar, value, ratio in zip(
    bars,
    cluster_size["n_customers"],
    cluster_size["cluster_ratio_percent"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value}\n({ratio:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(output_dir / "01_cluster_size_summary.png", dpi=300)
plt.show()

## 4. 시각화에 사용할 feature matrix

기본적으로 Step 4에서 선택된 feature set을 사용합니다. 현재 기본 선택 모델은 `axis_6_without_capacity_KMeans_k6`이므로 일반적으로 아래 6개 행동 축이 사용됩니다.

- `Activity_Score`
- `Decline_Risk_Score`
- `Inactivity_Score`
- `Relationship_Strength_Score`
- `Contact_Burden_Score`
- `Credit_Utilization_Score`

만약 weighted model을 선택한 경우에는 시각화용 행렬에도 동일한 가중치를 적용합니다.

In [ ]:
# ============================================================
# 5. 시각화용 feature matrix 생성
# ============================================================

FEATURE_WEIGHTS = {
    "Activity_Score": 1.2,
    "Decline_Risk_Score": 1.4,
    "Inactivity_Score": 1.2,
    "Relationship_Strength_Score": 1.0,
    "Contact_Burden_Score": 1.1,
    "Credit_Capacity_Score": 0.7,
    "Credit_Utilization_Score": 1.0,
}

X_original = df_vis[selected_features].copy()

# Weighted_KMeans로 선택된 경우, Step 4와 동일한 feature weighting을 적용
if "Weighted_KMeans" in selected_key or selected_key.startswith("weighted_"):
    X_model_space = X_original.copy()
    for col, weight in FEATURE_WEIGHTS.items():
        if col in X_model_space.columns:
            X_model_space[col] = X_model_space[col] * weight
    print("Weighted feature space를 사용합니다.")
else:
    X_model_space = X_original.copy()
    print("Unweighted selected feature space를 사용합니다.")

print("X_model_space:", X_model_space.shape)
display(X_model_space.head())

## 5. 2D PCA Scatter

PCA는 선형 투영 방식이므로 UMAP보다 덜 극적으로 보일 수 있지만, 투영 방식이 명확하고 설명하기 쉽습니다. 2D PCA는 cluster가 전반적으로 어떤 방향으로 나뉘는지 확인하는 보조 시각화입니다.

In [ ]:
# ============================================================
# 6. 2D PCA Scatter
# ============================================================

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
coords_2d = pca_2d.fit_transform(X_model_space)

df_pca_2d = df_vis.copy()
df_pca_2d["PC1"] = coords_2d[:, 0]
df_pca_2d["PC2"] = coords_2d[:, 1]

pca_2d_info = {
    "selected_key": selected_key,
    "features": selected_features,
    "explained_variance_ratio": pca_2d.explained_variance_ratio_.tolist(),
    "cumulative_explained_variance": float(pca_2d.explained_variance_ratio_.sum()),
}
save_json(pca_2d_info, output_dir / "02_pca_2d_info.json")

print("2D PCA explained variance ratio:", pca_2d.explained_variance_ratio_)
print("2D PCA cumulative explained variance:", pca_2d.explained_variance_ratio_.sum())

hover_cols = [
    "Row_ID",
    "Cluster",
    "Attrition_Label",
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Avg_Utilization_Ratio",
    "High_Risk_Signal_Count",
]
hover_cols = [col for col in hover_cols if col in df_pca_2d.columns]

fig = px.scatter(
    df_pca_2d,
    x="PC1",
    y="PC2",
    color="Cluster_Label",
    hover_data=hover_cols,
    title=(
        "2D PCA Scatter of Selected Clusters "
        f"(Explained Variance: {pca_2d.explained_variance_ratio_.sum():.1%})"
    ),
)

fig.update_traces(marker=dict(size=5, opacity=0.65))
fig.update_layout(**get_plotly_layout())
fig.show()
fig.write_html(output_dir / "02_2d_pca_scatter.html")

# Matplotlib static version
plt.figure(figsize=(8, 6))
for cluster_label, group in df_pca_2d.groupby("Cluster_Label"):
    plt.scatter(group["PC1"], group["PC2"], s=10, alpha=0.6, label=cluster_label)
plt.title("2D PCA Scatter of Selected Clusters")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(output_dir / "02_2d_pca_scatter.png", dpi=300)
plt.show()

## 6. 3D PCA Scatter

3D PCA는 선택된 feature space의 전체 구조를 3차원으로 압축해 보여줍니다. PCA 축은 직접적인 비즈니스 의미를 갖지는 않으므로, 해석은 보조적으로만 사용합니다.

In [ ]:
# ============================================================
# 7. 3D PCA Scatter
# ============================================================

pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
coords_3d = pca_3d.fit_transform(X_model_space)

df_pca_3d = df_vis.copy()
df_pca_3d["PC1"] = coords_3d[:, 0]
df_pca_3d["PC2"] = coords_3d[:, 1]
df_pca_3d["PC3"] = coords_3d[:, 2]

pca_3d_info = {
    "selected_key": selected_key,
    "features": selected_features,
    "explained_variance_ratio": pca_3d.explained_variance_ratio_.tolist(),
    "cumulative_explained_variance": float(pca_3d.explained_variance_ratio_.sum()),
    "note": "PCA axes are linear projections of selected clustering features and should be interpreted only as diagnostic visualization axes.",
}
save_json(pca_3d_info, output_dir / "03_pca_3d_info.json")

print("3D PCA explained variance ratio:", pca_3d.explained_variance_ratio_)
print("3D PCA cumulative explained variance:", pca_3d.explained_variance_ratio_.sum())

fig = px.scatter_3d(
    df_pca_3d,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Cluster_Label",
    hover_data=hover_cols,
    title=(
        "3D PCA Scatter of Selected Clusters "
        f"(Explained Variance: {pca_3d.explained_variance_ratio_.sum():.1%})"
    ),
)

fig.update_traces(marker=dict(size=3, opacity=0.65))
fig.update_layout(**get_plotly_layout())
fig.show()
fig.write_html(output_dir / "03_3d_pca_scatter.html")

## 7. 3D Direct Risk View

이 시각화는 PCA와 달리 축 자체가 해석 가능합니다.

- x축: `Activity_Score`
- y축: `Decline_Risk_Score`
- z축: `Inactivity_Score`

즉, 고객군이 활동성, 거래 감소 위험, 비활성 신호 측면에서 어떻게 배치되는지 확인합니다.

In [ ]:
# ============================================================
# 8. 3D Direct Risk View
# ============================================================

risk_view_features = [
    "Activity_Score",
    "Decline_Risk_Score",
    "Inactivity_Score",
]

if all(col in df_vis.columns for col in risk_view_features):
    fig = px.scatter_3d(
        df_vis,
        x="Activity_Score",
        y="Decline_Risk_Score",
        z="Inactivity_Score",
        color="Cluster_Label",
        hover_data=hover_cols,
        title="3D Direct Risk View: Activity × Decline Risk × Inactivity",
        labels={
            "Activity_Score": "Activity Score",
            "Decline_Risk_Score": "Decline Risk Score",
            "Inactivity_Score": "Inactivity Score",
        },
    )
    fig.update_traces(marker=dict(size=3, opacity=0.65))
    fig.update_layout(**get_plotly_layout())
    fig.show()
    fig.write_html(output_dir / "04_3d_direct_risk_view.html")
else:
    print("Risk view에 필요한 feature가 없어 건너뜁니다:", risk_view_features)

## 8. 3D Direct Behavior View

두 번째 직접 축 시각화는 활동성, 접촉 부담, 한도 사용률을 함께 봅니다.

- x축: `Activity_Score`
- y축: `Contact_Burden_Score`
- z축: `Credit_Utilization_Score`

이 그림은 거래 활동성과 고객 접촉 부담, 한도 사용 행동이 서로 어떻게 다른 방향의 행동 신호인지 확인하는 데 사용합니다.

In [ ]:
# ============================================================
# 9. 3D Direct Behavior View
# ============================================================

behavior_view_features = [
    "Activity_Score",
    "Contact_Burden_Score",
    "Credit_Utilization_Score",
]

if all(col in df_vis.columns for col in behavior_view_features):
    fig = px.scatter_3d(
        df_vis,
        x="Activity_Score",
        y="Contact_Burden_Score",
        z="Credit_Utilization_Score",
        color="Cluster_Label",
        hover_data=hover_cols,
        title="3D Direct Behavior View: Activity × Contact Burden × Credit Utilization",
        labels={
            "Activity_Score": "Activity Score",
            "Contact_Burden_Score": "Contact Burden Score",
            "Credit_Utilization_Score": "Credit Utilization Score",
        },
    )
    fig.update_traces(marker=dict(size=3, opacity=0.65))
    fig.update_layout(**get_plotly_layout())
    fig.show()
    fig.write_html(output_dir / "05_3d_direct_behavior_view.html")
else:
    print("Behavior view에 필요한 feature가 없어 건너뜁니다:", behavior_view_features)

## 9. 3D LDA Projection

LDA projection은 이미 도출된 cluster label을 기준으로 cluster 간 차이가 잘 보이도록 투영합니다.

따라서 이 그림은 **자연 군집이 완벽히 분리되어 있다는 증거**가 아니라, Step 4에서 선택된 cluster label 간 차이를 강조해서 보여주는 **보조 시각화**입니다.

In [ ]:
# ============================================================
# 10. 3D LDA Projection
# ============================================================

n_clusters = df_vis["Cluster"].nunique()
n_features = X_model_space.shape[1]
n_components = min(3, n_clusters - 1, n_features)

if n_components < 2:
    print("LDA projection을 수행하기 위한 component 수가 부족합니다.")
else:
    lda = LinearDiscriminantAnalysis(n_components=n_components)
    coords_lda = lda.fit_transform(X_model_space, df_vis["Cluster"])

    df_lda = df_vis.copy()
    df_lda["LD1"] = coords_lda[:, 0]
    df_lda["LD2"] = coords_lda[:, 1]
    if n_components >= 3:
        df_lda["LD3"] = coords_lda[:, 2]

    lda_info = {
        "selected_key": selected_key,
        "features": selected_features,
        "n_components": int(n_components),
        "explained_variance_ratio": lda.explained_variance_ratio_.tolist(),
        "cumulative_explained_variance": float(lda.explained_variance_ratio_.sum()),
        "note": (
            "LDA projection uses cluster labels to emphasize between-cluster differences. "
            "It should be interpreted as a supervised visualization of derived clusters, "
            "not as evidence that natural clusters are perfectly separated."
        ),
    }
    save_json(lda_info, output_dir / "06_lda_projection_info.json")

    print("LDA explained variance ratio:", lda.explained_variance_ratio_)
    print("LDA cumulative explained variance:", lda.explained_variance_ratio_.sum())

    if n_components >= 3:
        fig = px.scatter_3d(
            df_lda,
            x="LD1",
            y="LD2",
            z="LD3",
            color="Cluster_Label",
            hover_data=hover_cols,
            title="3D LDA Projection of Selected Clusters",
        )
        fig.update_traces(marker=dict(size=3, opacity=0.65))
        fig.update_layout(**get_plotly_layout())
        fig.show()
        fig.write_html(output_dir / "06_3d_lda_projection.html")
    else:
        fig = px.scatter(
            df_lda,
            x="LD1",
            y="LD2",
            color="Cluster_Label",
            hover_data=hover_cols,
            title="2D LDA Projection of Selected Clusters",
        )
        fig.update_traces(marker=dict(size=5, opacity=0.65))
        fig.update_layout(**get_plotly_layout())
        fig.show()
        fig.write_html(output_dir / "06_2d_lda_projection.html")

## 10. Step 4.5 결과 README 생성

마지막으로 이 단계에서 생성한 시각화 파일의 의미를 정리한 README를 생성합니다.

In [ ]:
# ============================================================
# 11. README 및 metadata 저장
# ============================================================

visualization_metadata = {
    "step": "Step 4.5 Clustering Visualization / Diagnostics",
    "selected_key": selected_key,
    "selected_features": selected_features,
    "input_files": {
        "selected_model": str(selected_model_path.relative_to(ROOT)),
        "selected_labels": str(selected_labels_path.relative_to(ROOT)),
        "df_cluster_input_used": str(df_cluster_path.relative_to(ROOT)),
        "df_profile_table_used": str(df_profile_path.relative_to(ROOT)),
    },
    "output_dir": str(output_dir.relative_to(ROOT)),
    "important_note": (
        "This step visualizes the selected clustering structure. "
        "It does not perform final segment profiling or strategy interpretation. "
        "UMAP is intentionally excluded."
    ),
}

save_json(visualization_metadata, output_dir / "00_visualization_run_metadata.json")

readme_text = f"""# Step 4.5 Clustering Visualization / Diagnostics

## 목적

이 폴더는 Step 4에서 선택한 최종 clustering 결과를 시각적으로 점검하기 위한 결과물을 저장합니다.

선택 모델: `{selected_key}`

## 입력 파일

- `{selected_model_path.relative_to(ROOT)}`
- `{selected_labels_path.relative_to(ROOT)}`
- `{df_cluster_path.relative_to(ROOT)}`
- `{df_profile_path.relative_to(ROOT)}`

## 주요 출력 파일

| 파일 | 설명 |
|---|---|
| `01_cluster_size_summary.csv` | cluster별 고객 수와 비율 |
| `01_cluster_size_summary.png` | cluster size bar chart |
| `02_2d_pca_scatter.html` | 2D PCA 기반 cluster 구조 시각화 |
| `02_2d_pca_scatter.png` | 2D PCA 정적 이미지 |
| `03_3d_pca_scatter.html` | 3D PCA 기반 cluster 구조 시각화 |
| `04_3d_direct_risk_view.html` | Activity × Decline Risk × Inactivity 직접 축 시각화 |
| `05_3d_direct_behavior_view.html` | Activity × Contact Burden × Credit Utilization 직접 축 시각화 |
| `06_3d_lda_projection.html` | cluster label 간 차이를 강조한 LDA projection |
| `selected_model_visualization_table.csv` | 시각화용 결합 테이블 |

## 해석 주의

PCA와 direct feature view는 cluster 구조를 점검하기 위한 보조 시각화입니다.
LDA projection은 이미 도출된 cluster label을 기준으로 cluster 간 차이를 강조한 지도적 시각화이므로, 자연 군집이 완벽히 분리되어 있다는 증거로 해석하지 않습니다.

최종 segment 이름, 이탈률, 위험 신호, 전략 제안은 Step 5 Segment Profiling에서 수행합니다.
"""

readme_path = output_dir / "README_clustering_visualizations.md"
readme_path.write_text(readme_text, encoding="utf-8")

print("Step 4.5 outputs saved to:", output_dir.relative_to(ROOT))
print("Generated files:")
for path in sorted(output_dir.iterdir()):
    print("-", path.name)

## 11. 다음 단계

이 노트북 실행이 끝나면 Step 5로 넘어갑니다.

Step 5에서는 다음 작업을 수행합니다.

1. 최종 cluster label을 profile table에 결합
2. cluster별 행동 축 평균 계산
3. cluster profile heatmap 생성
4. segment별 이탈률 계산
5. high-risk signal overlap 분석
6. 범주형 변수 분포 분석
7. segment 이름 확정
8. segment별 유지 전략 제안